# Perform feature selection on normalized data

## Import libraries

In [1]:
import os
import pathlib
import pprint

import pandas as pd
from pycytominer import feature_select
from pycytominer.cyto_utils import output
from timelapse_utils.file_utils.notebook_init_utils import (
    bandicoot_check,
    init_notebook,
)
from timelapse_utils.profiling_utils.sc_extraction_utils import add_single_cell_count_df

root_dir, in_notebook = init_notebook()
if in_notebook:
    import tqdm.notebook as tqdm
else:
    import tqdm

## Set paths and variables

In [2]:
# load in platemap file as a pandas dataframe
platemap_path = pathlib.Path(
    f"{root_dir}/Wave2_data/0.download_data/platemap/platemap.csv"
).resolve()

image_base_dir = bandicoot_check(
    bandicoot_mount_path=pathlib.Path(f"{os.path.expanduser('~')}/mnt/bandicoot/"),
    root_dir=root_dir,
)
image_base_dir = pathlib.Path(
    f"{image_base_dir}/live_cell_timelapse_pyroptosis_project_data/processed_data/"
).resolve(strict=True)
normalized_profiles_path = pathlib.Path(
    f"{image_base_dir}/8.normalized_profiles/normalized_profiles.parquet"
).resolve(strict=True)

feature_selected_profiles_path = pathlib.Path(
    f"{image_base_dir}/9.feature_selected_profiles/feature_selected_profiles.parquet"
).resolve()
feature_selected_profiles_path.parent.mkdir(exist_ok=True)

## Define dict of paths

## Perform feature selection

In [3]:
# define operations to be performed on the data
# list of operations for feature select function to use on input profile
feature_select_ops = [
    "variance_threshold",
    "blocklist",
    "drop_na_columns",
    "correlation_threshold",
]

This last cell does not get run due to memory constraints. 
It is run on an HPC cluster with more memory available.

In [4]:
# read in the annotated file
normalized_df = pd.read_parquet(normalized_profiles_path)
metadata_cols = [x for x in normalized_df.columns if x.startswith("Metadata_")]
normalized_features_df = normalized_df.drop(metadata_cols, axis="columns")
# perform feature selection with the operations specified
feature_select_df = feature_select(
    normalized_features_df,
    operation=feature_select_ops,
)

# add metadata columns back to the feature selected df
feature_select_df = pd.concat(
    [normalized_df[metadata_cols], feature_select_df], axis="columns"
)
print("Feature selection complete, saving to parquet file!")
# save features selected df as parquet file
output(
    df=feature_select_df,
    output_filename=feature_selected_profiles_path,
    output_type="parquet",
)
# check to see if the shape of the df has changed indicating feature selection occurred
print(feature_select_df.shape)
feature_select_df.head()

Feature selection complete, saving to parquet file!
(4627, 1029)


,Metadata_Well,Metadata_Inducer,Metadata_Inducer_dose,Metadata_Inhibitor,Metadata_Inhibitor_dose,Metadata_Time,Metadata_Well_FOV,Metadata_FOV,Metadata_Well_FOV_Time,Metadata_ImageNumber,...,Nuclei_Texture_DifferenceEntropy_BF_3_03_256,Nuclei_Texture_DifferenceVariance_BF_3_00_256,Nuclei_Texture_DifferenceVariance_BF_3_01_256,Nuclei_Texture_DifferenceVariance_BF_3_02_256,Nuclei_Texture_DifferenceVariance_BF_3_03_256,Nuclei_Texture_InverseDifferenceMoment_BF_3_01_256,Nuclei_Texture_InverseDifferenceMoment_BF_3_02_256,Nuclei_Texture_InverseDifferenceMoment_CL640_3_00_256,Nuclei_Texture_SumEntropy_BF_3_03_256,Nuclei_Texture_SumVariance_BF_3_03_256
0,B2,DMSO,0.15%,DMSO,0.15%,1,B2_1,1,B2_1_1,1,...,-0.356450,0.002784,0.444384,0.913946,0.172310,1.570404,1.741143,0.362593,-0.123765,1.367633
1,B2,DMSO,0.15%,DMSO,0.15%,1,B2_1,1,B2_1_1,1,...,-0.862200,0.866786,2.585464,1.017615,0.621810,2.251244,0.521666,1.014023,-1.227941,-1.194693
2,B2,DMSO,0.15%,DMSO,0.15%,1,B2_1,1,B2_1_1,1,...,1.933462,-1.687852,-1.640262,-1.242963,-1.458041,-2.657663,-3.049757,-0.304710,2.864955,3.337820
3,B2,DMSO,0.15%,DMSO,0.15%,1,B2_1,1,B2_1_1,1,...,-0.816553,-0.239516,-0.836549,0.259220,0.271893,-1.355415,0.115242,1.361821,0.981633,1.241597
4,B2,DMSO,0.15%,DMSO,0.15%,1,B2_1,1,B2_1_1,1,...,-0.308765,0.835496,0.770252,-0.188897,-0.055141,0.025439,-1.302815,1.367261,0.638412,0.254384
